# Usertally - Scattering of He ions from C foil

The `UserTally` option is employed to study small angle scattering of 270 keV He ions by a 100μg/cm$^2$ C foil.

Multile scattering of the ions at small angles is taken into account by the Monte-Carlo algorithm.
For this we lower the minimum scattering angle to 0.1$^\circ$ (default is 2$^\circ$).

Repeat the simulation with different number of ions (10$^4$, 10$^5$, 10$^6$) to see the effect on the error bars.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import opentrim

In [ ]:

# Ion beam: 270 keV He
config.IonBeam.ion = opentrim.Element('He')
config.IonBeam.energy_distribution.center = 270000  # eV

# Variable flight path - important for light ions
config.Transport.flight_path_type = opentrim.FlightPath.Variable
config.Transport.min_scattering_angle = 0.1 # allow calculation of low angle scattering

# Target: 100μg/cm2 C = 440 nm thick foil
mat = opentrim.Material()
mat.id = 'Carbon foil'
mat.density = 2.27
atom = opentrim.Atom()
atom.element = opentrim.Element('C')
atom.X = 1.0
atom.Ed = 20.0
atom.El = 3.0
atom.Es = 3.0
atom.Er = 20.0
atom.Rc = 0.946
mat.composition.append(atom)
config.Target.materials.append(mat)

region = opentrim.Region()
region.id = 'slab'
region.material_id = 'Carbon foil'
region.origin = [0, 0, 0]
region.size = [440, 1000, 1000]
config.Target.regions.append(region)

config.Target.cell_count = [1, 1, 1] # just 1 cell
config.Target.size = [440, 1000, 1000]

# Physics
config.Simulation.electronic_stopping = opentrim.Stopping.SRIM13
config.Simulation.simulation_type = opentrim.SimulationType.FullCascade

# UserTally
ut = opentrim.UserTally()
ut.id = "dPdOmega"
ut.description = "Angular distribution of He ions exiting the foil"
ut.event = opentrim.Event.IonExit
# prepare bins for the scattering angle θ
theta = np.linspace(10,0,21) # θ = 10 deg ... 0 deg, 20 bins
ut.bins.nx = np.cos(np.deg2rad(theta)) # convert to nx = cos(θ)
ut.bins.x = [440, 450] # only ions exiting at the back side
ut.bins.atom_id = [0, 1] # only beam atoms (id=0)
config.UserTally.append(ut)

# Run
config.Output.outfilename = 'He_270keV_C'
config.Run.max_no_ions = 10000
config.Run.threads = 4

rng = np.random.default_rng()
config.Run.seed = int(rng.random()*999999 + 1) # random seed

config.validate()
config

In [ ]:
sim = opentrim.Driver()
sim.init(config)

sim.run()                       # returns immediately
sim.wait()                      # block until done
print('done, total ions:', sim.ion_count())

info = opentrim.Info(sim)
info


In [ ]:
y, dy = info["user_tally"]["dPdOmega"]["data"]
nx = info["user_tally"]["dPdOmega"]["bins"]["1"]
delta_cos = nx[1:] - nx[:-1]
S = y[0, :, 0] / delta_cos / (4.0 * np.pi) # scattering probability per solid angle ΔΩ=Δcosθ
dS = 2*dy[0, :, 0] / delta_cos / (4.0 * np.pi) ## 2 x sigma - 90% confidence interval

plt.figure()
plt.errorbar(0.5*(theta[0:-1]+theta[1:]), S, yerr=dS, marker="o")
plt.yscale('log')
plt.xlabel("Scattering angle (deg)")
plt.ylabel("Particles per solid angle (srad$^{-1}$)")
plt.tight_layout()
plt.xlim(0,10)
plt.title("Scattering of 270keV He by 100μg/cm$^2$ C foil")
plt.show()